In [ ]:
!pip -q install chromadb langchain openai tiktoken

In [ ]:
!pip install -U langchain-community
!pip install -U langchain-openai
!pip install -U langchain-chroma
!pip install -U chromadb

In [ ]:
!pip install -U sentence-transformers langchain-huggingface

In [ ]:
!pip show tiktoken

In [ ]:

!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip   #to download the zip

In [ ]:
!unzip -q new_articles.zip -d new_articles #unzip the file

setting environment


In [ ]:
import os


Importing Libraries


In [ ]:
!pip show langchain

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [ ]:
loader = DirectoryLoader("/content/new_articles/",glob = "./*.txt",loader_cls = TextLoader)

In [ ]:
document = loader.load()

Dividing Data Into chunks

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = text_splitter.split_documents(document)

Creating Db


In [ ]:
from langchain_chroma import Chroma

persist_directory = "db"

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [ ]:
print(vectordb._collection.count())

In [ ]:
query = "What is Google I/O?"

results = vectordb.similarity_search(
    query=query,
    k=3
)

In [ ]:
for i, doc in enumerate(results, 1):
    print(f"Result {i}")
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 80)

Creating a Retriever


In [ ]:
retriever = vectordb.as_retriever()

docs = retriever.invoke(
    "How much money did Microsoft raise?"
)

In [ ]:
for doc in docs:
    print(doc.page_content)
    print(doc.metadata)
    print("=" * 80)

In [ ]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 2}
)

print("Search Type:", retriever.search_type)
print("Search Parameters:", retriever.search_kwargs)